# SS Encoder/Decoder Round-Trip Demo

This notebook loads GLB meshes, voxelizes them, runs them through the Sparse Structure VAE encoder and decoder, and visualizes the reconstruction.

In [ ]:
import sys
import os

import torch
import numpy as np
import trimesh
import matplotlib.pyplot as plt
from pathlib import Path

from sam3d_objects.model.backbone.tdfy_dit.models.sparse_structure_vae import (
    SparseStructureEncoderTdfyWrapper,
    SparseStructureDecoderTdfyWrapper,
)

# ============ CONFIG — edit these paths ============
GLB_PATHS = [
    "YOUR GLB PATH"
]

PROJECT_ROOT = "./sam-3d-objects"
ENCODER_CKPT = os.path.join(PROJECT_ROOT, "checkpoints/hf_weights/hf_weights/ss_encoder.ckpt")
DECODER_CKPT = os.path.join(PROJECT_ROOT, "checkpoints/hf_weights/hf_weights/ss_decoder.ckpt")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "notebook/output_ss_demo")
DEVICE = "cuda:0"
RESOLUTION = 64

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Device: {DEVICE}")
print(f"Resolution: {RESOLUTION}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"GLB files: {len(GLB_PATHS)}")

In [ ]:
! which python

In [ ]:
import torch
print(f"torch {torch.__version__}, CUDA: {torch.cuda.is_available()}")


In [ ]:
def normalize_mesh_verts(verts):
    """Normalize mesh vertices to [-0.5, 0.5]^3."""
    vmin = verts.min(axis=0)
    vmax = verts.max(axis=0)
    center = (vmax + vmin) / 2.0
    extent = vmax - vmin
    max_extent = np.max(extent)
    if max_extent == 0:
        vertices = verts - center
        scale = 1
    else:
        scale = 1.0 / max_extent
        vertices = (verts - center) * scale
    return vertices, scale, center


def glb_to_voxels(glb_path, resolution=64, save_ply_path=None):
    """
    Load a GLB file, voxelize it, and return an occupancy tensor.

    Args:
        glb_path: Path to the GLB file.
        resolution: Voxel grid resolution (default 64).
        save_ply_path: If set, save the normalized mesh as PLY.

    Returns:
        occupancy: Tensor of shape [1, resolution, resolution, resolution]
        voxel_coords: Numpy array of voxel center coordinates
    """
    # Load GLB
    scene_or_mesh = trimesh.load(glb_path)
    if isinstance(scene_or_mesh, trimesh.Scene):
        mesh = scene_or_mesh.dump(concatenate=True)
    else:
        mesh = scene_or_mesh

    verts = np.asarray(mesh.vertices)

    # Y-up → Z-up rotation: (x, y, z) → (x, z, -y)
    rot_matrix = np.array([[1, 0, 0], [0, 0, -1], [0, 1, 0]], dtype=np.float64)
    verts = verts @ rot_matrix.T

    # Normalize to [-0.5, 0.5]^3
    verts, scale, center = normalize_mesh_verts(verts)

    # Clamp to avoid boundary issues
    verts = np.clip(verts, -0.5 + 1e-6, 0.5 - 1e-6)

    # Update mesh vertices
    mesh.vertices = verts

    # Save normalized mesh if requested
    if save_ply_path is not None:
        mesh.export(save_ply_path)
        print(f"  Saved normalized mesh to {save_ply_path}")

    # Voxelize using trimesh
    pitch = 1.0 / resolution
    voxel_grid = mesh.voxelized(pitch)
    # Fill interior voxels for watertight meshes, fall back to surface-only
    try:
        voxel_grid = voxel_grid.fill(method="holes")
    except Exception:
        pass

    # Extract voxel coordinates (centers of occupied voxels)
    voxel_coords = voxel_grid.points

    # Build occupancy tensor from voxel grid indices
    sparse_indices = voxel_grid.sparse_indices  # [N, 3] integer grid indices
    sparse_indices = np.clip(sparse_indices, 0, resolution - 1)
    occupancy = torch.zeros(1, resolution, resolution, resolution, dtype=torch.float32)
    occupancy[:, sparse_indices[:, 0], sparse_indices[:, 1], sparse_indices[:, 2]] = 1.0

    return occupancy, voxel_coords


print("Helper functions defined.")

In [ ]:
# Load Encoder (config from ss_encoder.yaml)
encoder = SparseStructureEncoderTdfyWrapper(
    return_raw=True,
    in_channels=1,
    latent_channels=8,
    channels=[32, 128, 512],
    num_res_blocks=2,
    num_res_blocks_middle=2,
    pretrained_ckpt_path=ENCODER_CKPT,
)
encoder = encoder.to(DEVICE).eval()
print(f"Encoder loaded from {ENCODER_CKPT}")
print(f"  Parameters: {sum(p.numel() for p in encoder.parameters()):,}")

# Load Decoder (config from ss_decoder.yaml)
decoder = SparseStructureDecoderTdfyWrapper(
    out_channels=1,
    latent_channels=8,
    channels=[512, 128, 32],
    num_res_blocks=2,
    num_res_blocks_middle=2,
    reshape_input_to_cube=False,
    pretrained_ckpt_path=DECODER_CKPT,
)
decoder = decoder.to(DEVICE).eval()
print(f"Decoder loaded from {DECODER_CKPT}")
print(f"  Parameters: {sum(p.numel() for p in decoder.parameters()):,}")

In [ ]:
# Run voxelization on all GLB files
assert len(GLB_PATHS) > 0, "Please add GLB file paths to GLB_PATHS in the config cell."

voxel_results = {}  # name -> (occupancy, voxel_coords)

for glb_path in GLB_PATHS:
    name = Path(glb_path).stem
    print(f"\nProcessing: {name}")
    ply_path = os.path.join(OUTPUT_DIR, f"{name}_normalized.ply")
    occupancy, voxel_coords = glb_to_voxels(glb_path, resolution=RESOLUTION, save_ply_path=ply_path)
    voxel_results[name] = (occupancy, voxel_coords)
    num_voxels = int(occupancy.sum().item())
    print(f"  Occupancy shape: {occupancy.shape}")
    print(f"  Occupied voxels: {num_voxels} / {RESOLUTION**3} ({100*num_voxels/RESOLUTION**3:.1f}%)")

In [ ]:
# Encode → Decode round-trip
recon_results = {}  # name -> (input_occ, recon_occ, latent_stats)

for name, (occupancy, _) in voxel_results.items():
    print(f"\n=== {name} ===")
    input_voxels = occupancy.unsqueeze(0).to(DEVICE)  # [1, 1, 64, 64, 64]
    print(f"Input shape: {input_voxels.shape}")

    with torch.no_grad():
        # Encode
        enc_out = encoder(input_voxels)
        z, mean, logvar = enc_out["z"], enc_out["mean"], enc_out["logvar"]
        print(f"Latent z shape: {z.shape}")
        print(f"Latent mean — mean: {mean.mean().item():.4f}, std: {mean.std().item():.4f}, "
              f"min: {mean.min().item():.4f}, max: {mean.max().item():.4f}")

        # Decode from mean
        logits = decoder(mean)
        print(f"Decoder output shape: {logits.shape}")

        # Threshold
        recon_occ = (torch.sigmoid(logits) > 0.5).float()

    # Compute IoU
    input_bool = input_voxels.bool().cpu()
    recon_bool = recon_occ.bool().cpu()
    intersection = (input_bool & recon_bool).sum().item()
    union = (input_bool | recon_bool).sum().item()
    iou = intersection / union if union > 0 else 0.0
    print(f"IoU: {iou:.4f}  (intersection={intersection}, union={union})")

    recon_results[name] = (
        occupancy,           # [1, 64, 64, 64]
        recon_occ[0, 0].cpu(),  # [64, 64, 64]
        {"z_shape": list(z.shape), "iou": iou},
    )

In [ ]:
# Visualization: side-by-side 3D scatter plots and PLY saving

for name, (input_occ, recon_occ, stats) in recon_results.items():
    # Get occupied voxel centers
    input_coords = torch.nonzero(input_occ[0], as_tuple=False).numpy()  # [N, 3]
    recon_coords = torch.nonzero(recon_occ, as_tuple=False).numpy()     # [M, 3]

    # Convert grid indices to world coords: (idx + 0.5) / 64 - 0.5
    input_pts = (input_coords + 0.5) / RESOLUTION - 0.5
    recon_pts = (recon_coords + 0.5) / RESOLUTION - 0.5

    # Save as PLY using trimesh
    input_ply_path = os.path.join(OUTPUT_DIR, f"{name}_input_voxels.ply")
    recon_ply_path = os.path.join(OUTPUT_DIR, f"{name}_recon_voxels.ply")

    trimesh.PointCloud(input_pts).export(input_ply_path)
    trimesh.PointCloud(recon_pts).export(recon_ply_path)

    print(f"Saved: {input_ply_path}, {recon_ply_path}")

    # Matplotlib 3D scatter
    fig = plt.figure(figsize=(14, 6))

    ax1 = fig.add_subplot(121, projection="3d")
    ax1.scatter(input_pts[:, 0], input_pts[:, 1], input_pts[:, 2], s=1, alpha=0.5)
    ax1.set_title(f"{name} — Input ({len(input_pts)} voxels)")
    ax1.set_xlim(-0.5, 0.5); ax1.set_ylim(-0.5, 0.5); ax1.set_zlim(-0.5, 0.5)

    ax2 = fig.add_subplot(122, projection="3d")
    ax2.scatter(recon_pts[:, 0], recon_pts[:, 1], recon_pts[:, 2], s=1, alpha=0.5, c="orange")
    ax2.set_title(f"{name} — Reconstructed ({len(recon_pts)} voxels, IoU={stats['iou']:.3f})")
    ax2.set_xlim(-0.5, 0.5); ax2.set_ylim(-0.5, 0.5); ax2.set_zlim(-0.5, 0.5)

    plt.tight_layout()
    plt.show()